In [41]:
import os
import joblib
import numpy as np

import torch
import torch.nn as nn

## Load Pre-trained Models

In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SPAM_MODEL_PATH = "../models/nn_model.pth"
SPAM_VECTORIZER_PATH = "../models/nn_vectorizer.joblib"

PHISH_MODEL_PATH = "../models/phishing_nn_model.pth"
PHISH_VECTORIZER_PATH = "../models/phishing_nn_vectorizer.joblib"
PHISH_ENCODER_PATH = "../models/phishing_nn_encoder.joblib"

In [ ]:
class SpamFeedForwardNet(nn.Module):
    """
    Binary spam vs ham NN.
    Must match the architecture used in ff_nn.ipynb for the spam model.
    """
    def __init__(self, input_dim, hidden_dim=128, dropout_p=0.3):
        super().__init__()
        self.neural_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.neural_net(x)


class PhishTypeFeedForwardNet(nn.Module):
    """
    Multiclass phishing-type NN.
    The last layer outputs num_classes logits.
    This must match the architecture used in your phishing-type training notebook.
    """
    def __init__(self, input_dim, num_classes, hidden_dim=128, dropout_p=0.3):
        super().__init__()
        self.neural_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.neural_net(x)

In [44]:
def load_spam_nn_model(model_path, vectorizer_path):
    # Load vectorizer
    vectorizer = joblib.load(vectorizer_path)

    # Build model with correct input_dim
    input_dim = len(vectorizer.get_feature_names_out())
    model = SpamFeedForwardNet(input_dim=input_dim).to(device)

    # Load weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    return model, vectorizer


def load_phish_type_nn_model(model_path, vectorizer_path, encoder_path):
    # Load vectorizer and label encoder
    vectorizer = joblib.load(vectorizer_path)
    label_encoder = joblib.load(encoder_path)

    num_classes = len(label_encoder.classes_)

    # Build model with correct input_dim and num_classes
    input_dim = len(vectorizer.get_feature_names_out())
    model = PhishTypeFeedForwardNet(
        input_dim=input_dim,
        num_classes=num_classes,
    ).to(device)

    # Load weights
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    return model, vectorizer, label_encoder


# Load all models and helpers
spam_model, spam_vectorizer = load_spam_nn_model(SPAM_MODEL_PATH, SPAM_VECTORIZER_PATH)
phish_model, phish_vectorizer, phish_label_encoder = load_phish_type_nn_model(
    PHISH_MODEL_PATH,
    PHISH_VECTORIZER_PATH,
    PHISH_ENCODER_PATH,
)

## Test the Pipeline with Examples

In [45]:
def classify_email_nn(
    email_text: str,
    spam_threshold: float = 0.5,
    top_k: int = 3,
):
    """
    Two-stage NN pipeline:
      Stage 1: Spam vs Ham (binary)
      Stage 2: Phishing Type (multiclass) - ONLY if classified as spam
    """

    # --- Stage 1: Spam vs Ham ---
    X_spam = spam_vectorizer.transform([email_text])
    X_spam = torch.tensor(X_spam.toarray(), dtype=torch.float32).to(device)

    with torch.no_grad():
        spam_logits = spam_model(X_spam)
        spam_prob = torch.sigmoid(spam_logits).item()

    is_spam = spam_prob >= spam_threshold

    # Base result structure
    result = {
        "email_text": email_text.strip(),
        "spam_probability": spam_prob,
        "is_spam": is_spam,
        # Stage 2 phishing-type fields:
        "phishing_type": None,
        "phishing_type_confidence": None,
        "phishing_top_types": [],
        "final_label": None,
    }

    # If not spam, we stop here
    if not is_spam:
        result["final_label"] = "Ham (Not spam)"
        return result

    # --- Stage 2: Phishing Type (multiclass) ---
    X_phish = phish_vectorizer.transform([email_text])
    X_phish = torch.tensor(X_phish.toarray(), dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = phish_model(X_phish)          # shape (1, num_classes)
        probs = torch.softmax(logits, dim=1)   # shape (1, num_classes)
        probs_np = probs.cpu().numpy().reshape(-1)

    # Top-1 prediction
    pred_idx = int(probs_np.argmax())
    pred_label = phish_label_encoder.classes_[pred_idx]
    pred_conf = float(probs_np[pred_idx])

    # Top-k predictions
    sorted_indices = probs_np.argsort()[::-1]
    top_indices = sorted_indices[:top_k]

    top_types = []
    for i in range(len(top_indices)):
        idx = int(top_indices[i])
        class_name = phish_label_encoder.classes_[idx]
        prob = float(probs_np[idx])
        top_types.append(
            {
                "rank": i + 1,
                "class_name": class_name,
                "probability": prob,
            }
        )

    result["phishing_type"] = pred_label
    result["phishing_type_confidence"] = pred_conf
    result["phishing_top_types"] = top_types
    result["final_label"] = f"Phishing – {pred_label}"

    return result


## Batch Classification

In [46]:
def display_result_nn(result):
    print("=" * 60)
    print("EMAIL PREVIEW:\n")
    print(result["email_text"][:500], "\n")

    # Stage 1
    print(f"Stage 1 - Spam Probability: {result['spam_probability']:.3f}")
    print(f"Classified as Spam: {result['is_spam']}")

    # If not spam, we stop here
    if not result["is_spam"]:
        print("\nStage 2 - Phishing Type: skipped (email not classified as spam).")
        print(f"\nFINAL LABEL: {result['final_label']}")
        print("=" * 60)
        return

    # Stage 2 - Phishing Type
    print("\nStage 2 - Phishing Type (NN):")
    if result["phishing_type"] is None:
        print("  [No phishing type prediction available]")
    else:
        print(
            f"  Predicted type: {result['phishing_type']} "
            f"(confidence: {result['phishing_type_confidence']:.3f})"
        )

        if result["phishing_top_types"]:
            print("\n  Top candidates:")
            for item in result["phishing_top_types"]:
                print(
                    f"    {item['rank']}. {item['class_name']:25s} "
                    f"{item['probability']:.3f}"
                )

    print(f"\nFINAL LABEL: {result['final_label']}")
    print("=" * 60)


## Interactive Classification (Enter Your Own Email)

In [47]:
test_email = """
URGENT: Your account has been compromised!

We noticed suspicious login attempts. Click the link below to verify
your credentials and secure your account:

[Verify Now]

If you do not act within 24 hours, your account will be locked.
"""

res = classify_email_nn(test_email)
display_result_nn(res)


EMAIL PREVIEW:

URGENT: Your account has been compromised!

We noticed suspicious login attempts. Click the link below to verify
your credentials and secure your account:

[Verify Now]

If you do not act within 24 hours, your account will be locked. 

Stage 1 - Spam Probability: 1.000
Classified as Spam: True

Stage 2 - Phishing Type (NN):
  Predicted type: credential_harvesting (confidence: 0.320)

  Top candidates:
    1. credential_harvesting     0.320
    2. social_engineering        0.295
    3. urgency                   0.194

FINAL LABEL: Phishing – credential_harvesting


In [48]:
test_email = """
Dear Hiring Manager,

I'm writing to express my interest in the [job title] position advertised on your company's website. Please find my cover letter and résumé attached below. I'm excited to contribute my skills and experience to your team.

[Brief cover letter content highlighting relevant qualifications.]

Thank you for considering my application. I'd love to talk with you more about the position.

Sincerely,

[Your full name]
"""

res = classify_email_nn(test_email)
display_result_nn(res)

EMAIL PREVIEW:

Dear Hiring Manager,

I'm writing to express my interest in the [job title] position advertised on your company's website. Please find my cover letter and résumé attached below. I'm excited to contribute my skills and experience to your team.

[Brief cover letter content highlighting relevant qualifications.]

Thank you for considering my application. I'd love to talk with you more about the position.

Sincerely,

[Your full name] 

Stage 1 - Spam Probability: 0.000
Classified as Spam: False

Stage 2 - Phishing Type: skipped (email not classified as spam).

FINAL LABEL: Ham (Not spam)
